# 03. Model Training
**Goal:** rank each user's previously bought products so the top 10 form the "Buy it again" row.

**Input:** `model_dataset.parquet` from notebook 02 (train/validation/test split by user).

**Metric:** recall@10 = share of the user's reordered products that appear in our top 10. Users with no reorders are excluded.

**Rule:** all choices are made on validation; the test set is used once, at the end.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
import lightgbm as lgb

PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR/"data"/"raw"
PROCESSED_DATA_DIR = PROJECT_DIR/"data"/"processed"
REPORTS_DIR = PROJECT_DIR/"reports"

MODEL_DATASET_PATH = (PROCESSED_DATA_DIR/"model_dataset.parquet")

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

feature_columns = [
    "purchase_count",
    "purchase_share",
    "orders_since_last_purchase",
    "user_total_prior_orders",
    "user_avg_days_between_orders",
    "product_total_purchases",
    "product_reorder_rate"
]

due_columns = [
    "days_since_last_purchase",
    "avg_days_between_purchases",
    "due_ratio"
]
all_columns=feature_columns+due_columns

print("Model training environment initialized!")
print("Dataset exists:", MODEL_DATASET_PATH.exists())
print("LightGBM version: ", lgb.__version__)
print("Number of feaures: ", len(feature_columns), "| with due-ness length:", len(all_columns))

Model training environment initialized!
Dataset exists: True
LightGBM version:  4.7.0
Number of feaures:  7 | with due-ness length: 10


In [2]:
model_dataset_path = MODEL_DATASET_PATH.as_posix()

split_summary = con.execute(f"""
    SELECT split, COUNT(*) AS candidate_pairs, COUNT(DISTINCT user_id) AS unique_users,
    SUM(label) AS positive_labels
    FROM read_parquet('{model_dataset_path}')
    GROUP BY split
    ORDER BY split
""").df()

display(split_summary)

,split,candidate_pairs,unique_users,positive_labels
0,test,1297732,19879,125329.0
1,train,5905950,91546,579248.0
2,validation,1270979,19784,124247.0


## 1. Load data
Loads all 10 feature columns. `avg_days_between_purchases` and `due_ratio` are ~60% NULL (products bought only once); LightGBM handles NULLs natively.

In [3]:
train_df = con.execute(f"""
    SELECT user_id, product_id, {",".join(all_columns)}, label
    FROM read_parquet('{model_dataset_path}')
    WHERE split='train'
""").df()

validation_df = con.execute(f"""
    SELECT user_id, product_id, {",".join(all_columns)}, label
    FROM read_parquet('{model_dataset_path}')
    WHERE split='validation'
""").df()

print("Training shape: ", train_df.shape)
print("Validation shape:", validation_df.shape)
display(train_df[due_columns].isna().mean().round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training shape:  (5905950, 13)
Validation shape: (1270979, 13)


days_since_last_purchase      0.000
avg_days_between_purchases    0.599
due_ratio                     0.603
dtype: float64

In [4]:
X_train = train_df[feature_columns]
X_val = validation_df[feature_columns]

y_train = train_df["label"].astype("int8")
y_val = validation_df["label"].astype("int8")

print("Training features: ", X_train.shape)
print("Validation features: ", X_val.shape)

print("\nTraining positive rate: ", round(y_train.mean()*100,2),"%")
print("\nValidation positive rate: ", round(y_val.mean()*100,2), "%")

print("\nMissing values in training features: ", X_train.isna().sum().sum())
print("Missing values in validation features: ", X_val.isna().sum().sum())


Training features:  (5905950, 7)
Validation features:  (1270979, 7)

Training positive rate:  9.81 %

Validation positive rate:  9.78 %

Missing values in training features:  0
Missing values in validation features:  0


## 2. LightGBM classifier (7 base features)
Predicts the probability that each candidate product is reordered. Early stopping on validation log loss picks the number of trees.

In [5]:
model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate= 0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model.fit(
    X_train,y_train, eval_X=X_val, eval_y=y_val, eval_metric="binary_logloss",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)
print("Best iteration: ", model.best_iteration_)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.25071
[200]	valid_0's binary_logloss: 0.25048
[300]	valid_0's binary_logloss: 0.250392
[400]	valid_0's binary_logloss: 0.250358
[500]	valid_0's binary_logloss: 0.250321
[600]	valid_0's binary_logloss: 0.250306
Early stopping, best iteration is:
[619]	valid_0's binary_logloss: 0.250297
Best iteration:  619


In [6]:
from sklearn.metrics import log_loss

val_probabilities = model.predict_proba(X_val)[:,1]
print("Validation log loss: ", round(log_loss(y_val, val_probabilities),6))
print("Predictions generated:", len(val_probabilities))

Validation log loss:  0.250297
Predictions generated: 1270979


In [7]:
val_results = validation_df[["user_id", "product_id", "label"]].copy()
val_results["predicted_probability"] = val_probabilities

val_results["rank"] = (
    val_results.groupby("user_id")["predicted_probability"].rank(method="first", ascending=False))

display(val_results.head())

,user_id,product_id,label,predicted_probability,rank
0,68264,46979,0,0.068989,18.0
1,173989,14947,0,0.254609,7.0
2,58405,23026,0,0.047709,41.0
3,4505,13575,0,0.041901,13.0
4,89410,41338,1,0.132452,10.0


## 3. Evaluation function
One function scores  every approach the same way, on the same users: recall@k and precision@k (divided by min(k, candidates), since users with fewer than k past products cannot fill the row). Checked against notebook 02: it reproduces the baseline recall@10 of 0.537 exactly.

In [8]:
def evaluate(df, rank_col, ks=(5,10)):
    data = df[["user_id", "label", rank_col]].copy()

    for k in ks:
        data[f"hit_at_{k}"] = data["label"]*(data[rank_col]<=k)

    per_user = data.groupby("user_id").agg(
        positives = ("label","sum"),
        candidates = ("label","size"),
        **{f"hits_at_{k}": (f"hit_at_{k}", "sum") for k in ks}
    )

    per_user = per_user[per_user["positives"]>0]
    results = {"Users": len(per_user)}
    for k in ks:
        results[f"recall_at_{k}"] = (per_user[f"hits_at_{k}"]/per_user["positives"]).mean()
        results[f"precision_at_{k}"] = (per_user[f"hits_at_{k}"]/np.minimum(k, per_user["candidates"])).mean()

    return pd.DataFrame([results])

### Baselines 
- **Most-frequent:** the user's most bought products, ties are broken by recency. The rule a real app would ship first and the one the model must beat.
- **Popularity:** globally best selling products, as a floor.

In [9]:
eval_df = validation_df[["user_id","product_id","label","purchase_count","orders_since_last_purchase",
                         "product_total_purchases"]].copy()

eval_df = eval_df.sort_values(
    ["user_id", "purchase_count", "orders_since_last_purchase", "product_id"],
    ascending=[True, False, True, True])

eval_df["rank_frequency"] = eval_df.groupby("user_id").cumcount() + 1

eval_df = eval_df.sort_values(
    ["user_id", "product_total_purchases", "product_id"], ascending=[True, False, True])

eval_df["rank_popularity"]=eval_df.groupby("user_id").cumcount()+1

eval_df = eval_df.merge(
    val_results[["user_id", "product_id", "rank"]].rename(columns={"rank": "rank_lightgbm"}),
    on=["user_id", "product_id"],
    how="left"
)

print("Rows: ", len(eval_df))
print("Missing LightGBM ranks: ", eval_df["rank_lightgbm"].isna().sum())

Rows:  1270979
Missing LightGBM ranks:  0


In [10]:
results = pd.concat([
    evaluate(eval_df,"rank_popularity").assign(model="Popularity baseline (floor)"),
    evaluate(eval_df, "rank_frequency").assign(model="Most-frequent baseline"),
    evaluate(eval_df, "rank_lightgbm").assign(model="LightGBM classifier")
]).set_index("model")

display(results.round(4))

baseline_r10 = results.loc["Most-frequent baseline", "recall_at_10"]
model_r10 = results.loc["LightGBM classifier","recall_at_10"]
print(f"\nRecall@10: {baseline_r10:.4f} -> {model_r10:.4f}  "
      f"(+{(model_r10 - baseline_r10)*100:.2f} points, "
      f"{(model_r10/baseline_r10-1)*100:.1f}% relative)")

results.to_csv(REPORTS_DIR/"validation_results.csv")

,Users,recall_at_5,precision_at_5,recall_at_10,precision_at_10
model,,,,,
Popularity baseline (floor),18455,0.2340,0.2252,0.3804,0.1968
Most-frequent baseline,18455,0.3742,0.3800,0.5374,0.3015
LightGBM classifier,18455,0.3997,0.4060,0.5744,0.3240



Recall@10: 0.5374 -> 0.5744  (+3.71 points, 6.9% relative)


### Feature Importance (gain)
Measured by **gain** (total improvement each feature contributes), not split count, which favours features with many distinct values. User-product features (recency, share, count) carry ~90% of the gain; `product_total_purchases` is used for the most splits but adds only ~1.6%.

In [11]:
importance = pd.DataFrame({
    "feature": feature_columns,
    "gain": model.booster_.feature_importance(importance_type="gain"),
    "splits": model.booster_.feature_importance(importance_type= "split")
})
importance["gain_pct"] = (importance["gain"]/importance["gain"].sum()*100).round(1)
display(importance.sort_values("gain",ascending=False).reset_index(drop=True))

,feature,gain,splits,gain_pct
0,orders_since_last_purchase,3.208858e+06,2657,45.5
1,purchase_share,2.188624e+06,2402,31.0
2,purchase_count,9.920331e+05,1293,14.1
3,product_reorder_rate,4.161650e+05,3423,5.9
4,product_total_purchases,1.162265e+05,3587,1.6
5,user_total_prior_orders,7.356542e+04,1950,1.0
6,user_avg_days_between_orders,5.445121e+04,3258,0.8


In [12]:
import joblib

MODEL_DIR = PROJECT_DIR/"models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR/"lgbm_classifier.pkl"
joblib.dump(model, model_path)

print("Model saved to: ", model_path)
print("Best iteration: ", model.best_iteration_)

Model saved to:  c:\grocery_reorder_prediction\models\lgbm_classifier.pkl
Best iteration:  619


In [13]:
import json
feature_config = {
    "features": feature_columns,
    "target": "label",
    "model_type": "LGBMClassifier",
    "params": model.get_params(),
    "best_iteration": model.best_iteration_,
    "validation_metrics": results.loc["LightGBM classifier"].to_dict(),
}

config_path = MODEL_DIR/"feature_config.json"

with open(config_path, "w") as f:
    json.dump(feature_config, f, indent=4,default=str)

print("Feature configuration saved to: ", config_path)

Feature configuration saved to:  c:\grocery_reorder_prediction\models\feature_config.json


## 4. Experiment: LightGBM ranker (lambdarank)
Learns the order of products within each user directly instead of predicting each one separately. Rows are sorted by user and grouped, so the model compares products of the same user.

In [14]:
train_sorted = train_df.sort_values("user_id").reset_index(drop=True)
val_sorted = validation_df.sort_values("user_id").reset_index(drop=True)

train_groups = train_sorted.groupby("user_id", sort=True).size().to_numpy()
val_groups = val_sorted.groupby("user_id", sort=True).size().to_numpy()

print("Train users: ", len(train_groups), "| rows: ", train_groups.sum())
print("Val users: ", len(val_groups), "| rows: ", val_groups.sum())

Train users:  91546 | rows:  5905950
Val users:  19784 | rows:  1270979


In [15]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    n_estimators=3000,
    learning_rate=0.05,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

ranker.fit(
    train_sorted[feature_columns], train_sorted["label"],
    group = train_groups,
    eval_set=[(val_sorted[feature_columns], val_sorted["label"])],
    eval_group=[val_groups],
    eval_at=[10],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)
print("Best iteration: ", ranker.best_iteration_)

c:\Users\MITALI GUPTA\anaconda3\envs\grocery-reorder\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's ndcg@10: 0.575254
[200]	valid_0's ndcg@10: 0.575905
Early stopping, best iteration is:
[150]	valid_0's ndcg@10: 0.575906
Best iteration:  150


In [16]:
val_sorted["ranker_score"] = ranker.predict(val_sorted[feature_columns])
val_sorted["rank"] = (
    val_sorted.groupby("user_id")["ranker_score"].rank(method="first", ascending=False))

In [17]:
eval_df = eval_df.drop(columns=["rank_ranker"], errors="ignore")

eval_df = eval_df.merge(
    val_sorted[["user_id","product_id","rank"]].rename(columns={"rank":"rank_ranker"}),
    on=["user_id","product_id"],
    how="left"
)
print("Rows:", len(eval_df))
print("Missing ranker ranks: ", eval_df["rank_ranker"].isna().sum())

Rows: 1270979
Missing ranker ranks:  0


In [18]:
results = pd.concat([
    evaluate(eval_df, "rank_popularity").assign(model="Popularity baseline (floor)"),
    evaluate(eval_df, "rank_frequency").assign(model="Most-frequent baseline"),
    evaluate(eval_df, "rank_lightgbm").assign(model="LightGBM classifier"),
    evaluate(eval_df, "rank_ranker").assign(model="LightGBM ranker")
]).set_index("model")

display(results.round(4))

baseline_r10 = results.loc["Most-frequent baseline", "recall_at_10"]
for name in ["LightGBM classifier", "LightGBM ranker"]:
    r10 = results.loc[name, "recall_at_10"]
    print(f"{name}: recall@10 {baseline_r10:.4f} -> {r10:.4f} "
          f"(+{(r10-baseline_r10)*100:.2f} points vs baseline)")

gap = (results.loc["LightGBM ranker", "recall_at_10"]
       - results.loc["LightGBM classifier", "recall_at_10"]) * 100
print(f"\nRanker vs Classifier: {gap:+.2f} points of recall@10")

results.to_csv(REPORTS_DIR/"validation_results.csv")

,Users,recall_at_5,precision_at_5,recall_at_10,precision_at_10
model,,,,,
Popularity baseline (floor),18455,0.2340,0.2252,0.3804,0.1968
Most-frequent baseline,18455,0.3742,0.3800,0.5374,0.3015
LightGBM classifier,18455,0.3997,0.4060,0.5744,0.3240
LightGBM ranker,18455,0.4001,0.4057,0.5737,0.3236


LightGBM classifier: recall@10 0.5374 -> 0.5744 (+3.71 points vs baseline)
LightGBM ranker: recall@10 0.5374 -> 0.5737 (+3.63 points vs baseline)

Ranker vs Classifier: -0.08 points of recall@10


**Result:** the ranker and classifier tie(recall@10 0.5737 vs 0.5744). The classifier is kept: it is simpler and outputs probabilities. With per-item features, sorting by probability already captures most of the ranking

In [19]:
del train_sorted, val_sorted 
import gc; gc.collect()
print("Ranker data freed from memory")

Ranker data freed from memory


## 5. Adding due-ness features
Same model and settings (`**model.get_params()`), plus 3 day-based timing features from notebook 02. Labels are taken from the same DataFrame as the features, so they cannot fall out of line.

In [20]:
model_due = lgb.LGBMClassifier(**model.get_params())

model_due.fit(
    train_df[all_columns], train_df["label"].astype("int8"),
    eval_X = validation_df[all_columns], eval_y=validation_df["label"].astype("int8"),
    eval_metric="binary_logloss",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)
print("Best iteration:", model_due.best_iteration_)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.247606
[200]	valid_0's binary_logloss: 0.247195
[300]	valid_0's binary_logloss: 0.247119
[400]	valid_0's binary_logloss: 0.24705
[500]	valid_0's binary_logloss: 0.247003
[600]	valid_0's binary_logloss: 0.246981
Early stopping, best iteration is:
[633]	valid_0's binary_logloss: 0.246969
Best iteration: 633


In [21]:
val_results_due = validation_df[["user_id","product_id"]].copy()
val_results_due["probability"]= model_due.predict_proba(validation_df[all_columns])[:,1]
val_results_due["rank_lightgbm_due"]=(
    val_results_due.groupby("user_id")["probability"].rank(method="first", ascending=False))

eval_df = eval_df.drop(columns=["rank_lightgbm_due"], errors="ignore")
eval_df = eval_df.merge(
    val_results_due[["user_id", "product_id", "rank_lightgbm_due"]],
    on=["user_id", "product_id"],
    how="left")
print("Missing due-ness ranks:", eval_df["rank_lightgbm_due"].isna().sum())

Missing due-ness ranks: 0


In [22]:
results = pd.concat([
    evaluate(eval_df, "rank_popularity").assign(model="Popularity baseline (floor)"),
    evaluate(eval_df, "rank_frequency").assign(model="Most-frequent baseline"),
    evaluate(eval_df, "rank_lightgbm").assign(model="LightGBM classifier"),
    evaluate(eval_df, "rank_ranker").assign(model = "LightGBM ranker"),
    evaluate(eval_df, "rank_lightgbm_due").assign(model="LightGBM classifier + due-ness"),
]).set_index("model")

display(results.round(4))

base = results.loc["Most-frequent baseline", "recall_at_10"]
before = results.loc["LightGBM classifier", "recall_at_10"]
after = results.loc["LightGBM classifier + due-ness", "recall_at_10"]

print(f"Due-ness effect: {before:.4f} -> {after:.4f} ({(after-before)*100:+.2f} points)")
print(f"Total vs baseline: {base:.4f} -> {after:.4f} ({(after-base)*100:+.2f} points)")

results.to_csv(REPORTS_DIR/"validation_results.csv")

,Users,recall_at_5,precision_at_5,recall_at_10,precision_at_10
model,,,,,
Popularity baseline (floor),18455,0.2340,0.2252,0.3804,0.1968
Most-frequent baseline,18455,0.3742,0.3800,0.5374,0.3015
LightGBM classifier,18455,0.3997,0.4060,0.5744,0.3240
LightGBM ranker,18455,0.4001,0.4057,0.5737,0.3236
LightGBM classifier + due-ness,18455,0.4055,0.4133,0.5791,0.3282


Due-ness effect: 0.5744 -> 0.5791 (+0.46 points)
Total vs baseline: 0.5374 -> 0.5791 (+4.17 points)


In [23]:
importance_due = pd.DataFrame({
    "feature": all_columns,
    "gain": model_due.booster_.feature_importance(importance_type="gain"),
})

importance_due["gain_pct"] = (importance_due["gain"]/importance_due["gain"].sum()*100).round(1)
display(importance_due.sort_values("gain", ascending=False).reset_index(drop=True))

,feature,gain,gain_pct
0,purchase_share,2.481934e+06,33.5
1,orders_since_last_purchase,2.145135e+06,28.9
2,days_since_last_purchase,7.738676e+05,10.4
3,due_ratio,6.259158e+05,8.4
4,avg_days_between_purchases,5.027203e+05,6.8
5,product_reorder_rate,3.904197e+05,5.3
6,purchase_count,2.582703e+05,3.5
7,product_total_purchases,1.080854e+05,1.5
8,user_avg_days_between_orders,8.554727e+04,1.2
9,user_total_prior_orders,4.593380e+04,0.6


### Is the due-ness gain real?
Paired per-user comparison on validation: +0.46 points recall@10, 95% CI [+0.32, +0.61]. 2,213 users improved and 1,648 got worse; 79% unchanged. Due-ness features carry ~26% of gain, partly taking over from `orders_since_last_purchase` (45%->29%), so much of the timing signal was already present but is now measured in days instead of orders.

In [24]:
def per_user_recall(df, rank_col, k=10):
    d = df[["user_id","label",rank_col]].copy()
    d["hit"]=d["label"]*(d[rank_col]<=k)
    g=d.groupby("user_id").agg(pos=("label","sum"),hits=("hit","sum"))
    g=g[g["pos"]>0]
    return g["hits"]/g["pos"]

r_before = per_user_recall(eval_df, "rank_lightgbm")
r_after = per_user_recall(eval_df, "rank_lightgbm_due")
diff = r_after-r_before

se = diff.std()/np.sqrt(len(diff))
print(f"Mean difference: {diff.mean()*100:+.2f} points")
print(f"95% interval: [{(diff.mean()-1.96*se)*100:+.2f}, {(diff.mean()+1.96*se)*100:+.2f}] points")
print("Users better:", (diff>0).sum(), "|worse:", (diff<0).sum(), "|same:", (diff==0).sum())

Mean difference: +0.46 points
95% interval: [+0.32, +0.61] points
Users better: 2213 |worse: 1648 |same: 14594


In [25]:
import joblib, json

model_due_path = MODEL_DIR/"lgbm_classifier_due.pkl"
joblib.dump(model_due, model_due_path)

due_config = {
    "features": all_columns,
    "target": "label",
    "model_type": "LGBMClassifier",
    "params": model_due.get_params(),
    "best_iteration": model_due.best_iteration_,
    "validation_metrics": results.loc["LightGBM classifier + due-ness"].to_dict(),
    "improvement_vs_classifier_recall10_points": round(diff.mean()*100,2),
    "improvement_95ci_points": [round((diff.mean()-1.96*se)*100,2),
                                round((diff.mean()+1.96*se)*100,2)]
}
with open(MODEL_DIR/"feature_config_due.json", "w") as f:
    json.dump(due_config, f, indent=4, default=str)

print("Saved:", model_due_path)

Saved: c:\grocery_reorder_prediction\models\lgbm_classifier_due.pkl


## 6. Hyperparameter tuning
8 settings tried on a 30% user sample (learning rate 0.1 for speed), varying `num_leaves` (15-255), `min_child_samples` (20-100) and `colsample_bytree` (0.6-1.0). Setting 1 is the current model's setting and serves as the reference.

**Result:** all settings reached validation recall@10 between 0.5777 and 0.5784, versus 0.5782 for the current setting, so the differences are within noise. Default settings were kept.

**Insight:** a 30% user sample nearly matches full-data performance (0.5782 vs 0.5791), and small and large trees perform the same. Results are limited by the features, not by model capacity or data volume.

In [26]:
import time

rng = np.random.default_rng(42);
train_users = train_df["user_id"].unique()
sample_users = rng.choice(train_users, size=int(len(train_users)*0.3), replace=False)
tune_df = train_df[train_df["user_id"].isin(sample_users)]
print("Tuning rows: ", len(tune_df), "| users: ", len(sample_users))

param_grid = [
    {"num_leaves":31, "min_child_samples":20, "colsample_bytree":0.8},
    {"num_leaves":63, "min_child_samples":20, "colsample_bytree":0.8},
    {"num_leaves":127, "min_child_samples":50, "colsample_bytree":0.8},
    {"num_leaves":63, "min_child_samples":100, "colsample_bytree":0.8},
    {"num_leaves":63, "min_child_samples":50, "colsample_bytree":0.6},
    {"num_leaves":255, "min_child_samples":100, "colsample_bytree":0.8},
    {"num_leaves":15, "min_child_samples":20, "colsample_bytree":1.0},
    {"num_leaves":63, "min_child_samples":50, "colsample_bytree":1.0},
]

base_params = model_due.get_params()
base_params["learning_rate"] = 0.1

tuning_rows = []
for i, p in enumerate(param_grid, start=1):
    start = time.time()
    m = lgb.LGBMClassifier(**{**base_params, **p})
    m.fit(
        tune_df[all_columns], tune_df["label"].astype("int8"),
        eval_X = validation_df[all_columns], eval_y = validation_df["label"].astype("int8"),
        eval_metric="binary_logloss",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    tmp=validation_df[["user_id", "label"]].copy()
    tmp["score"] = m.predict_proba(validation_df[all_columns])[:,1]
    tmp["rank"] = tmp.groupby("user_id")["score"].rank(method="first", ascending=False)
    r = evaluate(tmp, "rank")

    tuning_rows.append({
        **p,
        "best_iter": m.best_iteration_,
        "recall_at_5": r.loc[0, "recall_at_5"],
        "recall_at_10": r.loc[0, "recall_at_10"],
        "minutes": round((time.time()-start)/60,1),
    })

    print(f"{i}/{len(param_grid)} {p} -> recall@10 {r.loc[0, 'recall_at_10']:.4f}")

tuning_results = pd.DataFrame(tuning_rows).sort_values("recall_at_10", ascending=False)
display(tuning_results.round(4))
tuning_results.to_csv(REPORTS_DIR/"tuning_results.csv", index=False)

Tuning rows:  1773697 | users:  27463
1/8 {'num_leaves': 31, 'min_child_samples': 20, 'colsample_bytree': 0.8} -> recall@10 0.5782
2/8 {'num_leaves': 63, 'min_child_samples': 20, 'colsample_bytree': 0.8} -> recall@10 0.5783
3/8 {'num_leaves': 127, 'min_child_samples': 50, 'colsample_bytree': 0.8} -> recall@10 0.5784
4/8 {'num_leaves': 63, 'min_child_samples': 100, 'colsample_bytree': 0.8} -> recall@10 0.5777
5/8 {'num_leaves': 63, 'min_child_samples': 50, 'colsample_bytree': 0.6} -> recall@10 0.5782
6/8 {'num_leaves': 255, 'min_child_samples': 100, 'colsample_bytree': 0.8} -> recall@10 0.5779
7/8 {'num_leaves': 15, 'min_child_samples': 20, 'colsample_bytree': 1.0} -> recall@10 0.5780
8/8 {'num_leaves': 63, 'min_child_samples': 50, 'colsample_bytree': 1.0} -> recall@10 0.5782


,num_leaves,min_child_samples,colsample_bytree,best_iter,recall_at_5,recall_at_10,minutes
2,127,50,0.8,53,0.4046,0.5784,0.4
1,63,20,0.8,72,0.4043,0.5783,0.3
0,31,20,0.8,115,0.4050,0.5782,0.4
4,63,50,0.6,104,0.4048,0.5782,0.4
7,63,50,1.0,68,0.4055,0.5782,0.3
6,15,20,1.0,225,0.4054,0.5780,0.5
5,255,100,0.8,46,0.4050,0.5779,0.3
3,63,100,0.8,85,0.4054,0.5777,0.4


## 7. Final evaluation on the test set (run once)
19,879 users never used for training, early stopping, tuning or any decision.

In [27]:
test_df = con.execute(f"""
    SELECT user_id, product_id, {",".join(all_columns)}, label
    FROM read_parquet('{model_dataset_path}')    
    WHERE split='test'
""").df()

print("Test shape:", test_df.shape, "| users:", test_df["user_id"].nunique())

test_df["prob_classifier"] = model.predict_proba(test_df[feature_columns])[:,1]
test_df["prob_due"] = model_due.predict_proba(test_df[all_columns])[:,1]
test_df["rank_lightgbm"] = test_df.groupby("user_id")["prob_classifier"].rank(method="first", ascending=False)
test_df["rank_lightgbm_due"] = test_df.groupby("user_id")["prob_due"].rank(method="first", ascending=False)

test_df = test_df.sort_values(
    ["user_id", "purchase_count", "orders_since_last_purchase", "product_id"], 
    ascending=[True, False, True, True])
test_df["rank_frequency"] = test_df.groupby("user_id").cumcount()+1

test_df = test_df.sort_values(
    ["user_id", "product_total_purchases", "product_id"],
    ascending=[True, False, True])
test_df["rank_popularity"] = test_df.groupby("user_id").cumcount()+1

test_results = pd.concat([
    evaluate(test_df, "rank_popularity").assign(model="Popularity baseline (floor)"),
    evaluate(test_df, "rank_frequency").assign(model="Most-frequent baseline"),
    evaluate(test_df, "rank_lightgbm").assign(model="LightGBM classifier"),
    evaluate(test_df, "rank_lightgbm_due").assign(model="LightGBM classifier + due-ness final")
]).set_index("model")

display(test_results.round(4))
test_results.to_csv(REPORTS_DIR/"test_results.csv")

test_diff = per_user_recall(test_df, "rank_lightgbm_due") - per_user_recall(test_df,"rank_frequency")
test_se = test_diff.std()/np.sqrt(len(test_diff))
print(f"\nFinal vs most-frequent baseline (recall@10): {test_diff.mean()*100:+.2f} points, "
      f"95% CI [{(test_diff.mean()-1.96*test_se)*100:+.2f}, {(test_diff.mean()+1.96*test_se)*100:+.2f}]")

Test shape: (1297732, 13) | users: 19879


,Users,recall_at_5,precision_at_5,recall_at_10,precision_at_10
model,,,,,
Popularity baseline (floor),18575,0.2350,0.2259,0.3813,0.1974
Most-frequent baseline,18575,0.3728,0.3806,0.5362,0.3023
LightGBM classifier,18575,0.4005,0.4076,0.5750,0.3251
LightGBM classifier + due-ness final,18575,0.4068,0.4154,0.5801,0.3296



Final vs most-frequent baseline (recall@10): +4.38 points, 95% CI [+4.10, +4.67]


## Summary

|Test set (18,575 users)|Recall@10|Precision@10|
|---|---|---|
|Popularity baseline|0.381|0.197|
|Most-frequent baseline|0.536|0.302|
|LightGBM classifier|0.575|0.325|
|**+ due-ness (final)**|**0.580**|**0.330**|

- Final model beats the most-frequent baseline by **+4.38 points recall@10 (8.2% relative)**, 95% CI [+4.10, +4.67].
- Due-ness adds +0.46 (validation)/ +0.51 (test) points and is statistically significant.
- Ranker vs classifier: tie, so the classifier is kept.
- Tuning: no meaningful gain, results are limited by features, not model capacity or data volume.
- Validation and test agree closely, so there is no overfitting to validation.